In [1]:
!pip install langchain langchain-community weaviate-client sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 7.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.5/633.5 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.22.0
    Uninstalling zstandard-0.22.0:
      Successfully uninstalled zstandard-0.22.0


In [9]:
!pip install -U langchain-weaviate

/Users/jb/miniconda3/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=2421) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 11.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.4
    Uninstalling numpy-2.2.4:
      Successfully uninstalled numpy-2.2.4


In [ ]:
# declare name of the collection
COLLECTION_NAME = "QuestionAnswering"
QUERY_SENTENCE = "사이보그가 뭐야?"
NUM_OF_DATA = 200
WEIGHT_OF_VECTOR_IN_HYBRID = 0.5

In [ ]:
import pandas as pd
from datasets import load_dataset

DATA_SET    = "beomi/KoAlpaca-v1.1a"
DATA_FILE   = "./data/KoAlpaca-train.csv"

load_data = load_dataset(DATA_SET, split="train")
load_data.to_csv(DATA_FILE)
csv_data = pd.read_csv(DATA_FILE)
csv_data.head(5)
data_to_insert = csv_data.head(min(NUM_OF_DATA, len(load_data)))

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# KURE-v1 임베딩 모델 초기화
embeddings = HuggingFaceEmbeddings(
    model_name="nlpai-lab/KURE-v1",
    model_kwargs={'device': 'cpu'},  # GPU 사용 시 'cuda'로 변경
    encode_kwargs={'normalize_embeddings': True}  # 벡터 정규화 옵션
)

/var/folders/6q/ffdj7m4x1079ttl0dxw9p8hr0000gn/T/ipykernel_2421/904259930.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [20]:
import weaviate
from langchain_weaviate.vectorstores import WeaviateVectorStore

# 벡터 저장소 초기화
client = weaviate.connect_to_local(host='localhost', port=8080)
db = WeaviateVectorStore(
    client=client,
    index_name=COLLECTION_NAME,  # Weaviate 클래스 이름
    text_key="content",      # 주요 텍스트 필드
    embedding=embeddings,    # 임베딩 모델
)

# vector_store = Weaviate(
#     client=client,
#     index_name=COLLECTION_NAME,  # Weaviate 클래스 이름
#     text_key="content",      # 주요 텍스트 필드
#     embedding=embeddings,    # 임베딩 모델
#     by_text=False            # 외부 임베딩 사용
# )

/Users/jb/miniconda3/lib/python3.12/site-packages/weaviate/warnings.py:314: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(


In [23]:
import pandas as pd

# Load the CSV data
data_file = "./data/KoAlpaca-train.csv"
csv_data = pd.read_csv(data_file).head(100)
# 100 rows

# Combine 'instruction' and 'output' for embedding
texts = (csv_data["instruction"] + " " + csv_data["output"]).tolist()  # Concatenate instruction and output

# Prepare metadata (excluding 'instruction' and 'output')
metadatas = (csv_data
             # .drop(columns=["instruction", "output"])
             .to_dict(orient="records"))  # Other columns as metadata

# Add the combined data to the vector store
db.add_texts(texts=texts, metadatas=metadatas)

['f754af62-c802-4c50-aac6-67f6f0688b5b',
 '783b93f0-1905-42b2-9b07-9d834553dc11',
 'f3454852-016f-445b-a8e8-61f2c25a135d',
 '79d46c2d-de99-4673-9f32-16ff79ec65c7',
 '4d4a241c-f21e-43d1-8f62-96f001600da4',
 '0625da03-454e-42af-81ac-bbc5b3f53d27',
 '235455ee-8aa8-4517-8d6f-56aae6c9c69b',
 '5d37b8ae-afd2-44ca-a888-70cff2c00e62',
 '4c1e1637-64c5-4220-b177-f9025b3bed8f',
 'aec4672d-e933-479a-961d-e679c8eef37e',
 '95f2ef58-0b20-4a00-8da9-4098fc49b494',
 '95f635d8-b79d-49a5-ac8d-2baaa9ae4650',
 '65d82622-1250-49bb-aeb2-d30f4d75607a',
 '1c11b734-9a4f-4e89-a78e-b897597f7d14',
 'ab0d0f6d-3125-458b-9459-a5d2c9c485fd',
 '7f3ab6e4-bac3-4c89-b1df-cf996d765a71',
 '65c4537c-fe7d-4e4d-ab58-ec4576cc8b3a',
 'b9f92524-e736-47c4-bd20-d5db986e4e3b',
 'e12a9665-1625-4a84-ab52-dc59c34a655f',
 '776ce117-dee7-432e-bd98-e09f9c33e82b',
 'b2dc66d8-2de4-4389-be7b-b42247778415',
 'b6996077-6c4e-496f-9f7d-286111174bd0',
 '3764fbbe-b0ac-45ce-8c3d-10e614378d3f',
 '97276dff-c55f-47f8-b2be-7a2eba938583',
 '58b557a2-a64e-

In [25]:
response = db.similarity_search(
    query=QUERY_SENTENCE,
    k=2,  # 가져올 결과 수
)

for doc in response:
    print(f"내용: {doc.page_content}")
    print(f"instruction: {doc.metadata.get('instruction')}")
    print(f"output: {doc.metadata.get('output')}")
    print(f"url: {doc.metadata.get('url')}")
    print("---")

내용: '사이보그'는 언제 처음 등장한 말이며, 그 의미와 종류에는 어떤 것이 있는지 알고 싶습니다. '사이보그'는 1960년에 처음 등장한 말로, 기계와 유기체가 합성되어 생겨난 새로운 개념을 나타냅니다. 주로 공상과학영화에서 등장하며, 1985년 페미니스트인 도나 해러웨이가 발표한 '사이보그를 위한 선언문'으로 인해 사이보그는 성차별 사회를 극복하는 사회정치적 상징으로 더욱 주목받게 되었습니다. 사이보그는 생물체와 기술적 개입이 결합된 다양한 종류가 있으며, 이제는 인공장기를 받은 사람, 예방접종을 받은 사람, 또는 사람과 기계가 결합된 베리칩과 같은 기술적 개입을 받은 사람들까지 우리 주변에서 만날 수 있습니다. 

또한, 지구 위치 파악 기능이나 신체 건강 상태를 감지하는 센서 등을 추가적으로 부착하면서 사이보그의 구조와 기능이 더욱 다양해지고 있습니다. 이러한 발전은 사이보그의 대중화를 가속화시키고 있으나, 범죄나 인권 침해 등의 문제가 발생할 가능성도 존재합니다.
instruction: '사이보그'는 언제 처음 등장한 말이며, 그 의미와 종류에는 어떤 것이 있는지 알고 싶습니다.
output: '사이보그'는 1960년에 처음 등장한 말로, 기계와 유기체가 합성되어 생겨난 새로운 개념을 나타냅니다. 주로 공상과학영화에서 등장하며, 1985년 페미니스트인 도나 해러웨이가 발표한 '사이보그를 위한 선언문'으로 인해 사이보그는 성차별 사회를 극복하는 사회정치적 상징으로 더욱 주목받게 되었습니다. 사이보그는 생물체와 기술적 개입이 결합된 다양한 종류가 있으며, 이제는 인공장기를 받은 사람, 예방접종을 받은 사람, 또는 사람과 기계가 결합된 베리칩과 같은 기술적 개입을 받은 사람들까지 우리 주변에서 만날 수 있습니다. 

또한, 지구 위치 파악 기능이나 신체 건강 상태를 감지하는 센서 등을 추가적으로 부착하면서 사이보그의 구조와 기능이 더욱 다양해지고 있습니다. 이러한 발전은 사이보그의 대중화를 가속화시키고 있으나, 범죄나 인권 침해 등의 문제가 발생할 가능성

In [25]:
response = db.similarity_search(
    query=QUERY_SENTENCE,
    k=2,  # 가져올 결과 수
)

for doc in response:
    print(f"내용: {doc.page_content}")
    print(f"instruction: {doc.metadata.get('instruction')}")
    print(f"output: {doc.metadata.get('output')}")
    print(f"url: {doc.metadata.get('url')}")
    print("---")

내용: '사이보그'는 언제 처음 등장한 말이며, 그 의미와 종류에는 어떤 것이 있는지 알고 싶습니다. '사이보그'는 1960년에 처음 등장한 말로, 기계와 유기체가 합성되어 생겨난 새로운 개념을 나타냅니다. 주로 공상과학영화에서 등장하며, 1985년 페미니스트인 도나 해러웨이가 발표한 '사이보그를 위한 선언문'으로 인해 사이보그는 성차별 사회를 극복하는 사회정치적 상징으로 더욱 주목받게 되었습니다. 사이보그는 생물체와 기술적 개입이 결합된 다양한 종류가 있으며, 이제는 인공장기를 받은 사람, 예방접종을 받은 사람, 또는 사람과 기계가 결합된 베리칩과 같은 기술적 개입을 받은 사람들까지 우리 주변에서 만날 수 있습니다. 

또한, 지구 위치 파악 기능이나 신체 건강 상태를 감지하는 센서 등을 추가적으로 부착하면서 사이보그의 구조와 기능이 더욱 다양해지고 있습니다. 이러한 발전은 사이보그의 대중화를 가속화시키고 있으나, 범죄나 인권 침해 등의 문제가 발생할 가능성도 존재합니다.
instruction: '사이보그'는 언제 처음 등장한 말이며, 그 의미와 종류에는 어떤 것이 있는지 알고 싶습니다.
output: '사이보그'는 1960년에 처음 등장한 말로, 기계와 유기체가 합성되어 생겨난 새로운 개념을 나타냅니다. 주로 공상과학영화에서 등장하며, 1985년 페미니스트인 도나 해러웨이가 발표한 '사이보그를 위한 선언문'으로 인해 사이보그는 성차별 사회를 극복하는 사회정치적 상징으로 더욱 주목받게 되었습니다. 사이보그는 생물체와 기술적 개입이 결합된 다양한 종류가 있으며, 이제는 인공장기를 받은 사람, 예방접종을 받은 사람, 또는 사람과 기계가 결합된 베리칩과 같은 기술적 개입을 받은 사람들까지 우리 주변에서 만날 수 있습니다. 

또한, 지구 위치 파악 기능이나 신체 건강 상태를 감지하는 센서 등을 추가적으로 부착하면서 사이보그의 구조와 기능이 더욱 다양해지고 있습니다. 이러한 발전은 사이보그의 대중화를 가속화시키고 있으나, 범죄나 인권 침해 등의 문제가 발생할 가능성